<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/main/notebooks/ppo_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Force headless pygame BEFORE anything imports pygame (directly or
# transitively). SDL picks its video driver at import time, so
# setting this later is a no-op for an already-loaded SDL backend.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

# Reinforce Tactics — PPO Bootstrap Curriculum

Warm-start a `MaskablePPO` policy by walking it through a fixed curriculum of
`(map, opponent)` stages before handing the resulting checkpoint off to
self-play (`notebooks/ppo_training.ipynb`).

Six stages: starter map (random → simple → medium), then beginner map
(random → simple → medium). A stage advances only when its eval win-rate
stays at or above its `promotion_win_rate` for `patience` consecutive
evaluations. If a stage exhausts its `max_timesteps` budget without
promoting, the runner raises `CurriculumStalled` — fix the underlying
issue (reward shaping, hyperparams) before raising the budget.

Stage list lives in `configs/bootstrap.yaml`; edit there to tune
thresholds, budgets, or PPO hyperparams without touching this notebook.

**Runtime:** GPU recommended (L4 or better). CPU is possible but slower.

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q gymnasium stable-baselines3 sb3-contrib tensorboard pandas numpy torch matplotlib opencv-python-headless

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repo and install as a package
import os
import sys
from pathlib import Path

REPO_DIR = Path("reinforce-tactics")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
elif Path("notebooks").exists():
    # Already inside the repo
    os.chdir("..")
else:
    print("Cloning repository...")
    !git clone https://github.com/kuds/reinforce-tactics.git
    os.chdir(REPO_DIR)

# Install the package so all imports resolve
!pip install -q -e .

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")

## 2. Imports

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from sb3_contrib import MaskablePPO

from reinforcetactics.rl.bootstrap import (
    CurriculumStalled,
    load_bootstrap_config,
    run_curriculum,
)
from reinforcetactics.rl.evaluation import evaluate_model
from reinforcetactics.rl.masking import make_maskable_env

print("All imports successful.")

## 2b. Storage configuration

Choose where to save outputs. Set `USE_GOOGLE_DRIVE = True` to persist
results to Google Drive (recommended for Colab — files survive runtime
disconnects). Set `False` to use local/Colab storage.

Each execution saves outputs under a datetime-stamped subfolder
(e.g. `benchmarks/bootstrap/20250615_143022/`), so previous runs are
preserved automatically.

In [ ]:
USE_GOOGLE_DRIVE = True
DRIVE_BASE_DIR = "reinforce-tactics/benchmarks"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        SAVE_DIR = Path("/content/drive/MyDrive") / DRIVE_BASE_DIR
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Google Drive mounted successfully.")
        print(f"Save directory: {SAVE_DIR}")
    except ImportError:
        print("WARNING: google.colab not available (not running in Colab).")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
    except Exception as e:
        print(f"WARNING: Failed to mount Google Drive: {e}")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
else:
    SAVE_DIR = None
    print("Using local storage (default).")
    print("  Tip: Set USE_GOOGLE_DRIVE = True to persist results to Google Drive.")

## 3. Configuration

Reads `configs/bootstrap.yaml`. To experiment, copy that file, edit, and
point `CONFIG_PATH` at the copy.

Each stage can override `max_steps`, `max_turns`, and `ent_coef` to
handle map-size or exploration shifts (the bigger `beginner.csv` map
needs a longer turn budget; the first beginner stage gets an entropy
bump to break the starter-map policy out of its rut). The summary
table below highlights any active overrides per stage; absent values
inherit `cfg.env` / `cfg.ppo` defaults.

In [ ]:
CONFIG_PATH = Path("configs/bootstrap.yaml")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
if SAVE_DIR is not None:
    OUTPUT_DIR = SAVE_DIR / "bootstrap" / RUN_ID
else:
    OUTPUT_DIR = Path("benchmarks") / "bootstrap" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_bootstrap_config(CONFIG_PATH)

print(f"Config:        {CONFIG_PATH}")
print(f"Output dir:    {OUTPUT_DIR}")
print(f"Defaults:      max_steps={cfg.env.max_steps}, max_turns={cfg.env.max_turns}, ent_coef={cfg.ppo.ent_coef}")
print(f"Stages:        {len(cfg.stages)}")

# Per-stage table -- highlight per-stage overrides so the curriculum's
# map-transition handling (longer turn budget, entropy bump, reward
# reshape) is visible at a glance.
header = (
    f"  {'name':26s}  {'opp':16s}  {'WR>=':>5s}  "
    f"{'budget':>10s}  {'max_steps':>10s}  {'max_turns':>10s}  "
    f"{'ent_coef':>9s}  {'reward keys overridden'}"
)
print(header)
print("  " + "-" * (len(header) - 2))


def _fmt_override(value, default):
    if value is None:
        return f"({default})"
    return f"{value}*"


for s in cfg.stages:
    reward_keys = sorted((s.reward_config or {}).keys())
    reward_str = ", ".join(reward_keys) if reward_keys else "(default)"
    print(
        f"  {s.name:26s}  {s.opponent:16s}  "
        f"{s.promotion_win_rate:>4.0%}  {s.max_timesteps:>10,}  "
        f"{_fmt_override(s.max_steps, cfg.env.max_steps):>10s}  "
        f"{_fmt_override(s.max_turns, cfg.env.max_turns):>10s}  "
        f"{_fmt_override(s.ent_coef, cfg.ppo.ent_coef):>9s}  "
        f"{reward_str}"
    )
print("  (* = per-stage override; (n) = inherits cfg default)")

print(f"\nEval freq:     {cfg.eval_freq:,} env steps")
print(f"n_envs:        {cfg.n_envs}")
print(f"Total budget:  {sum(s.max_timesteps for s in cfg.stages):,} env steps (worst case)")
print(f"Storage:       {'Google Drive' if USE_GOOGLE_DRIVE else 'Local'}")

## 4. Run the curriculum

One `model.learn()` call per stage, with the env swapped between stages
via `model.set_env(...)`. The shared `MaskablePPO` instance survives the
whole run, so the policy carries forward; only the `(map, opponent)` pair
changes. Best-by-eval-win-rate checkpoints land in
`OUTPUT_DIR/<stage_name>/best_model.zip`; the post-promotion model is
written as `stage_final.zip`.

In [ ]:
try:
    result = run_curriculum(cfg, output_dir=OUTPUT_DIR)
    stalled = None
except CurriculumStalled as exc:
    print(f"\nSTALLED: {exc}")
    print("Investigate before raising the budget \u2014 a stuck stage usually")
    print("signals reward shaping or hyperparam issues, not insufficient steps.")
    result = None
    stalled = exc

if result is not None:
    print(f"\nFinal model: {result['final_model_path']}")
    for h in result["history"]:
        last = h["results"][-1] if h["results"] else None
        last_wr = f"{last['win_rate']:.1%}" if last else "n/a"
        print(f"  {h['stage']:18s}  promoted={h['promoted']!s:5s}  best_wr={h['best_win_rate']:.1%}  last_wr={last_wr}")

## 5. Win-rate over the full curriculum

Concatenates eval snapshots across stages onto a single timestep axis.
Vertical lines mark stage transitions; horizontal lines mark each stage's
promotion threshold. A regression bump right after a transition is
expected with hard cutoffs — the policy is encountering a new opponent
for the first time.

In [ ]:
if result is None:
    print("Skipped \u2014 no successful run to plot.")
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    cmap = plt.get_cmap("tab10")
    for i, h in enumerate(result["history"]):
        if not h["results"]:
            continue
        xs = [r["timesteps"] for r in h["results"]]
        ys = [r["win_rate"] for r in h["results"]]
        ax.plot(xs, ys, marker="o", label=h["stage"], color=cmap(i % 10))
        # Promotion threshold for the stage.
        stage_cfg = next(s for s in cfg.stages if s.name == h["stage"])
        ax.hlines(
            stage_cfg.promotion_win_rate,
            xmin=xs[0],
            xmax=xs[-1],
            colors=cmap(i % 10),
            linestyles=":",
            alpha=0.5,
        )
        # Stage transition marker at the last timestep of the stage.
        ax.axvline(xs[-1], color="gray", linestyle="--", alpha=0.3)
    ax.set_xlabel("Cumulative env timesteps")
    ax.set_ylabel("Eval win rate")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title("Curriculum win rate (dotted = stage threshold, dashed = transition)")
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=9)
    out = OUTPUT_DIR / "curriculum_winrate.png"
    fig.tight_layout()
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")

## 5b. Stage diagnostics: outcomes / length / reward mix / action mix

Four panels per stage (drawn separately so each stage's timestep axis
is locally meaningful):

1. **W/L/D over time** — `0/0/30` (all draws) means episodes are timing
   out without a winner; `0/30/0` (all losses) means the opponent is
   crushing the agent. Different fixes for each.
2. **Mean episode length** — saturating at `max_steps` confirms the
   turn cap is the rate-limiter; climbing then dropping is the policy
   learning to win earlier.
3. **Reward decomposition** — large green `action` band with ~0
   `terminal` band reads as 'farms shaping but never wins'. Growing
   orange `terminal` band is healthy convergence.
4. **Action mix** — dominant `end_turn` is policy collapse;
   `create_unit`-heavy with no `attack`/`seize` is over-economy.

In [ ]:
from reinforcetactics.rl.viz import (
    plot_action_distribution,
    plot_outcome_breakdown,
    plot_reward_decomposition,
)

if result is None:
    print("Skipped \u2014 no successful run to diagnose.")
else:
    diagnostics_dir = OUTPUT_DIR / "diagnostics"
    diagnostics_dir.mkdir(parents=True, exist_ok=True)

    for h in result["history"]:
        if not h["results"]:
            print(f"[{h['stage']}] no eval snapshots, skipping diagnostics")
            continue
        stage_charts_dir = diagnostics_dir / h["stage"]
        stage_charts_dir.mkdir(parents=True, exist_ok=True)

        # Mean episode length per eval -- a small standalone chart so
        # the saturating-at-cap pattern is unmistakable.
        results_seq = h["results"]
        timesteps = [r["timesteps"] for r in results_seq]
        lengths = [r["avg_length"] for r in results_seq]
        fig_len, ax_len = plt.subplots(figsize=(10, 3.2))
        ax_len.plot(timesteps, lengths, marker="o")
        ax_len.set_xlabel("Timesteps")
        ax_len.set_ylabel("Mean episode length")
        ax_len.set_title(f"[{h['stage']}] Episode length per eval")
        ax_len.grid(alpha=0.3)
        fig_len.tight_layout()
        fig_len.savefig(stage_charts_dir / "episode_length.png", dpi=120, bbox_inches="tight")
        plt.show()

        # The viz helpers expect ``.results``-shaped lists and write
        # their png next to whatever ``charts_dir`` we pass.
        plot_outcome_breakdown(results_seq, charts_dir=stage_charts_dir)
        plt.show()
        plot_reward_decomposition(results_seq, charts_dir=stage_charts_dir)
        plt.show()
        plot_action_distribution(results_seq, charts_dir=stage_charts_dir)
        plt.show()
        print(f"[{h['stage']}] diagnostics saved to {stage_charts_dir}")

## 6. Sanity eval on the final stage

Reload the saved checkpoint and run a fresh evaluation against the
hardest stage (beginner map vs medium bot) to confirm the exit criterion
really holds. Uses more episodes than the in-training eval for a tighter
estimate.

In [ ]:
if result is None:
    print("Skipped \u2014 no final checkpoint.")
else:
    final_stage = cfg.stages[-1]
    sanity_env = make_maskable_env(
        map_file=final_stage.map_file,
        opponent=final_stage.opponent,
        max_steps=cfg.env.max_steps,
        max_turns=cfg.env.max_turns,
        reward_config=cfg.env.reward_config,
        enabled_units=cfg.env.enabled_units,
        action_space_type=cfg.env.action_space_type,
        seed=cfg.seed + 9999,  # different seed than training eval
    )
    loaded = MaskablePPO.load(result["final_model_path"])
    metrics = evaluate_model(loaded, sanity_env, n_episodes=100, seed=cfg.seed + 9999)
    sanity_env.close()
    print(
        f"Sanity eval ({final_stage.name}, n=100):  "
        f"WR={metrics['win_rate']:.1%}  "
        f"reward={metrics['avg_reward']:+.1f}  "
        f"W/L/D={metrics['wins']}/{metrics['losses']}/{metrics['draws']}"
    )
    if metrics["win_rate"] < final_stage.promotion_win_rate:
        print(
            f"  WARNING: sanity WR {metrics['win_rate']:.1%} < threshold "
            f"{final_stage.promotion_win_rate:.1%}. The policy may be\n"
            "  overfit to the training eval seed. Consider a longer final\n"
            "  stage or rolling-window promotion before warm-starting self-play."
        )

## 7. Hand off to self-play

The final checkpoint is the artifact `notebooks/ppo_training.ipynb`'s
self-play setup should load. Drop this snippet into that notebook in
place of the fresh `MaskablePPO(...)` constructor call:

```python
from sb3_contrib import MaskablePPO

BOOTSTRAP_CHECKPOINT = '<paste path printed below>'
model = MaskablePPO.load(BOOTSTRAP_CHECKPOINT, env=self_play_vec_env)
```

`set_env()` is called implicitly by `MaskablePPO.load(..., env=...)`, so
the policy weights load while the env is the new self-play one.

In [ ]:
if result is not None:
    print("Hand-off path:")
    print(f"  {result['final_model_path']}")
    print("\nIn ppo_training.ipynb:")
    print(f"  BOOTSTRAP_CHECKPOINT = '{result['final_model_path']}'")
    print("  model = MaskablePPO.load(BOOTSTRAP_CHECKPOINT, env=self_play_vec_env)")

In [ ]:
import time

print("Training finished. Disconnecting runtime in 5 seconds...")
time.sleep(5)

from google.colab import runtime

runtime.unassign()